# Recent data inspection

What the data files of the latest update look like: **columns and types, number of observations, and the first rows**.
Every number comes from a parquet footer or from the first rows of a file, so no table is scanned: the notebook runs
on the login node in under a minute and needs less than 1 GB of memory.

| section | contents |
|---|---|
| 1 | every file under `<pipeline>/output*/` and `validation/data/` modified since `SINCE` |
| 2 | OpenAlex: `paper_metadata`, `paper_topics`, `paper_topics_long`, `paper_disruption`, two coverage CSVs |
| 3 | PatentView: `patent_inventor`, `patent_disruption` |
| 4 | Dimensions: `paper_author_country` |
| 5 | PATSTAT: the three output sets side by side, then every file of one set |
| 6 | `validation/data/inventory.json` (the dashboard's table inventory) |

`null_%` is the footer statistics' null count summed over row groups. It stays empty for list columns, whose leaf
null count mixes null lists, empty lists and null elements.

In [ ]:
import os, sys, glob, json
import datetime as dt
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display, Markdown

sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/validation')
import val_common as V

SINCE = '2026-09-14'      # a file modified on or after this date counts as updated
N_HEAD = 5                # rows shown per file

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 250)

In [ ]:
def summary(path):
    # size, modification time, rows and columns of one file: parquet from the footer, csv read whole (they are small)
    st = os.stat(path)
    row = {'file': os.path.relpath(path, V.BASE),
           'modified': dt.datetime.fromtimestamp(st.st_mtime).strftime('%Y-%m-%d %H:%M'),
           'size_GB': round(st.st_size / 1e9, 3)}
    if path.endswith('.parquet'):
        pf = pq.ParquetFile(path)
        row.update(rows=pf.metadata.num_rows, columns=len(pf.schema_arrow.names), row_groups=pf.metadata.num_row_groups)
    elif path.endswith('.csv'):
        df = pd.read_csv(path)
        row.update(rows=len(df), columns=df.shape[1])
    return row


def columns(path):
    # name, Arrow type and null share of every column
    pf = pq.ParquetFile(path)
    md, schema = pf.metadata, pf.schema_arrow
    leaf = {md.schema.column(i).path: i for i in range(md.num_columns)}   # a list column's leaf is '<name>.list.element'
    rows = []
    for name in schema.names:
        typ = schema.field(name).type
        nulls = md.num_rows if str(typ) == 'null' else None
        i = leaf.get(name)
        if nulls is None and i is not None:
            stats = [md.row_group(g).column(i).statistics for g in range(md.num_row_groups)]
            if all(s is not None and s.has_null_count for s in stats):
                nulls = sum(s.null_count for s in stats)
        rows.append({'column': name, 'type': str(typ),
                     'null_%': None if nulls is None else round(100 * nulls / max(md.num_rows, 1), 2)})
    return pd.DataFrame(rows)


def head(path, n=N_HEAD):
    # first n rows; only the first row group is read
    batch = next(pq.ParquetFile(path).iter_batches(batch_size=n), None)
    return pd.DataFrame() if batch is None else batch.to_pandas()


def show(path, n=N_HEAD):
    # summary line, column table and first rows of one parquet or csv file
    s = summary(path)
    display(Markdown(f"`{s['file']}`: **{s.get('rows', 0):,} rows x {s.get('columns', 0)} columns**, "
                     f"{s['size_GB']} GB, modified {s['modified']}"))
    if path.endswith('.parquet'):
        display(columns(path))
        display(head(path, n))
    else:
        display(pd.read_csv(path).head(n))

## 1. Files updated since `SINCE`

Every file under a pipeline's output folders (`<pipeline>/output*/`, sub-folders included) and under `validation/data/`
with a modification time on or after `SINCE`. `Atypicality/` is a separate project and is left out.

In [ ]:
since = dt.datetime.fromisoformat(SINCE).timestamp()
candidates = glob.glob(f'{V.BASE}/*/output*/**/*', recursive=True) + glob.glob(f'{V.VAL}/data/*')
paths = sorted(p for p in candidates
               if os.path.isfile(p) and not p.startswith(f'{V.BASE}/Atypicality/') and os.path.getmtime(p) >= since)
overview = pd.DataFrame([summary(p) for p in paths])
overview = overview.astype({c: 'Int64' for c in ('rows', 'columns', 'row_groups') if c in overview})
print(f'{len(overview)} files, {overview.size_GB.sum():.1f} GB, modified since {SINCE}')
overview

## 2. OpenAlex

### 2a. `paper_metadata.parquet`

One row per OpenAlex work. Rewritten 2026-10-01: new `primary_topic_topic` and the list columns `field` / `subfield` /
`topic` (all topics of the work, best score first); the other columns are identical to the previous file row for row.
`author_list` is all-null and kept only for schema compatibility; authors are in `paper_author.parquet`.

In [ ]:
show(V.paper('paper_metadata.parquet'))

### 2b. `paper_topics.parquet`

New (2026-09-22). One row per work: all scored topics with ids, names and scores, their subfields / fields / domains,
the primary topic, and the keywords. Multi-valued columns are `;`-joined strings in score order.

In [ ]:
show(V.paper('paper_topics.parquet'))

### 2c. `paper_topics_long.parquet`

New (2026-09-22). One row per work x topic; `rank` 1 is the best-scoring topic.

In [ ]:
show(V.paper('paper_topics_long.parquet'))

### 2d. `paper_disruption.parquet`

CD index per work for the 3-, 5-, 10-year and full windows. The columns from `pctl_year` on were appended 2026-09-14:
the percentile of each CD within publication year x first FoS (`pctl_group`), among works with a non-null CD.

In [ ]:
show(V.paper('paper_disruption.parquet'))

### 2e. Coverage tables

`paper_fos_coverage_by_year.csv` (share of works with a level-0 FoS, written by the CD-percentile job) and
`paper_topics_coverage_by_year.csv` (works with at least one topic, written by `paper_topics`).

In [ ]:
show(V.paper('paper_fos_coverage_by_year.csv'))
show(V.paper('paper_topics_coverage_by_year.csv'))

## 3. PatentView

### 3a. `patent_inventor.parquet`

New (2026-09-21). One row per utility patent with at least one disambiguated inventor: the PatentView counterpart of
`paper_author`. Use `n_inventors` for team size rather than splitting `inventor_list`, because a few inventor ids
contain a `;` themselves.

In [ ]:
show(V.patent('patent_inventor.parquet'))

### 3b. `patent_disruption.parquet`

The same CD columns as the papers; percentiles within filing year x CPC Section (`pctl_group`), appended 2026-09-14.

In [ ]:
show(V.patent('patent_disruption.parquet'))

## 4. Dimensions

### 4a. `paper_author_country.parquet`

New (2026-10-01). One row per Dimensions publication, with the same 9 columns as
`OpenAlex/output/paper_author_country.parquet`.

In [ ]:
show(V.dim('paper_author_country.parquet'))

## 5. PATSTAT

Three output sets with the same 15 file names, all built 2026-10-01 (details in `PATSTAT/Readme.txt`):

| folder | unit | year |
|---|---|---|
| `output/` | application (`appln_id`) | filing year |
| `output_grant/` | granted application | grant year |
| `output_family/` | DOCDB family (`docdb_family_id`) | earliest priority year |

### 5a. The three sets side by side

Rows per file in each set, then, for each file, the columns of a set that are not in all three sets.

In [ ]:
PS_SETS = {'output': V.PS_OUT, 'output_grant': V.PS_OUT_GRANT, 'output_family': V.PS_OUT_FAMILY}
ps_files = sorted({os.path.basename(p) for d in PS_SETS.values() for p in glob.glob(f'{d}/*.parquet')})
ps_schema = {(s, f): pq.read_schema(f'{d}/{f}').names
             for s, d in PS_SETS.items() for f in ps_files if os.path.exists(f'{d}/{f}')}
ps_rows = pd.DataFrame({s: {f: pq.ParquetFile(f'{d}/{f}').metadata.num_rows
                            for f in ps_files if os.path.exists(f'{d}/{f}')} for s, d in PS_SETS.items()})
display(ps_rows.style.format('{:,.0f}', na_rep='-'))

# per file and set: the columns that the other sets do not all have (keys and clock columns differ by design)
diff = {}
for f in ps_files:
    cols = {s: ps_schema.get((s, f), []) for s in PS_SETS}
    shared = set.intersection(*(set(c) for c in cols.values()))
    diff[f] = {s: ', '.join(x for x in c if x not in shared) for s, c in cols.items()}
pd.DataFrame(diff).T

### 5b. Every file of one set

`PS_DIR` picks the set: `V.PS_OUT` (applications, filing year), `V.PS_OUT_GRANT` or `V.PS_OUT_FAMILY`.

In [ ]:
PS_DIR = V.PS_OUT

#### 5b.1 `patstat_metadata.parquet`

In [ ]:
show(f'{PS_DIR}/patstat_metadata.parquet')

#### 5b.2 `patstat_reference.parquet`

In [ ]:
show(f'{PS_DIR}/patstat_reference.parquet')

#### 5b.3 `patstat_citation.parquet`

In [ ]:
show(f'{PS_DIR}/patstat_citation.parquet')

#### 5b.4 `patstat_citation_trend.parquet`

In [ ]:
show(f'{PS_DIR}/patstat_citation_trend.parquet')

#### 5b.5 `patstat_uniqueC_trend.parquet`

In [ ]:
show(f'{PS_DIR}/patstat_uniqueC_trend.parquet')

#### 5b.6 `patstat_hit_probability.parquet`

In [ ]:
show(f'{PS_DIR}/patstat_hit_probability.parquet')

#### 5b.7 `patstat_sb.parquet`

In [ ]:
show(f'{PS_DIR}/patstat_sb.parquet')

#### 5b.8 `patstat_disruption.parquet`

In [ ]:
show(f'{PS_DIR}/patstat_disruption.parquet')

#### 5b.9 `patstat_disruption_trend.parquet`

In [ ]:
show(f'{PS_DIR}/patstat_disruption_trend.parquet')

#### 5b.10 `patstat_disruption_trend_summary.parquet`

In [ ]:
show(f'{PS_DIR}/patstat_disruption_trend_summary.parquet')

#### 5b.11 `patstat_feg_disruption_trend.parquet`

In [ ]:
show(f'{PS_DIR}/patstat_feg_disruption_trend.parquet')

#### 5b.12 `patstat_z_score.parquet`

In [ ]:
show(f'{PS_DIR}/patstat_z_score.parquet')

#### 5b.13 `z_score_pair.parquet`

In [ ]:
show(f'{PS_DIR}/z_score_pair.parquet')

#### 5b.14 `patstat_inventor.parquet`

In [ ]:
show(f'{PS_DIR}/patstat_inventor.parquet')

#### 5b.15 `patstat_inventor_country.parquet`

In [ ]:
show(f'{PS_DIR}/patstat_inventor_country.parquet')

## 6. `validation/data/inventory.json`

The dashboard's table inventory, rebuilt 2026-10-01 from parquet footers: the top-level fields, then the first entries
of its file list.

In [ ]:
inv = json.load(open(f'{V.VAL}/data/inventory.json'))
display(pd.Series({k: v for k, v in inv.items() if not isinstance(v, (list, dict))}, name='value').to_frame())
inv_files = pd.DataFrame(inv['files'])
print(f'{len(inv_files)} entries')
inv_files.head(N_HEAD)